In [ ]:
from pyspark.sql.functions import lit, date_format, current_timestamp
from datetime import datetime
from pyspark.sql.types import StringType


In [ ]:
from delta.tables import DeltaTable


In [ ]:
%run 2. Parameters

In [ ]:
# Step 1: Create the enhanced dim_database table with parameter metadata
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {lakehouse_name}.dim_database (
    DATABASE_NAME STRING,
    DATABASE_VERSION STRING,
    WORKSPACE_ID STRING,
    DATASET_ID STRING,
    SEMANTIC_MODEL_NAME STRING,
    DATA_AGENT_NAME STRING,
    MODEL_NAME STRING,
    LAKEHOUSE_NAME STRING,
    TABLE_SCHEMA STRING,
    LOAD_TIMESTAMP STRING
)
USING DELTA
""")

print("Table 'dim_database' created successfully with parameter columns")

In [ ]:
# Step 1: Cache frequently accessed parameter values
workspace_id = workspaceId
dataset_id = datasetId
load_ts = f"{datetime.now():%d.%m.%Y %H:%M:%S}"

# Step 2: Read tables with only required columns (projection pushdown)
required_cols = ["DATABASE_NAME", "DATABASE_VERSION"]

df_data_sources = spark.table(f"{lakehouse_name}.data_sources").select(required_cols)
df_agent_dax = spark.table(f"{lakehouse_name}.agent_dax_documentation").select(required_cols)
df_column_metadata = spark.table(f"{lakehouse_name}.column_metadata").select(required_cols)

# Step 3: Union and distinct in one go (more efficient)
df_dim_database = df_data_sources \
    .union(df_agent_dax) \
    .union(df_column_metadata) \
    .distinct()

# Step 4: Add columns in a single transformation (reduces DataFrame operations)
df_dim_database = df_dim_database \
    .withColumn("WORKSPACE_ID", lit(workspace_id)) \
    .withColumn("DATASET_ID", lit(dataset_id)) \
    .withColumn("SEMANTIC_MODEL_NAME", lit(semantic_model_name)) \
    .withColumn("DATA_AGENT_NAME", lit(data_agent_name)) \
    .withColumn("MODEL_NAME", lit(model_name)) \
    .withColumn("LAKEHOUSE_NAME", lit(lakehouse_name)) \
    .withColumn("TABLE_SCHEMA", lit(table_schema)) \
    .withColumn("LOAD_TIMESTAMP", lit(load_ts))

# Step 5: Cache if you'll use this DataFrame multiple times
df_dim_database.cache()

# Show preview
display(df_dim_database)
print(f"Total distinct database records: {df_dim_database.count()}")

# Step 6: Optimized merge operation
target_table = f"{lakehouse_name}.dim_database"

if spark.catalog.tableExists(target_table):
    deltaTable = DeltaTable.forName(spark, target_table)
    
    # Use more efficient merge with explicit column list
    deltaTable.alias("target").merge(
        df_dim_database.alias("source"),
        "target.DATABASE_NAME = source.DATABASE_NAME AND target.DATABASE_VERSION = source.DATABASE_VERSION"
    ).whenNotMatchedInsertAll() \
    .execute()
    
    print("New data merged into 'dim_database' successfully")
else:
    df_dim_database.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)
    
    print("Table 'dim_database' created and data inserted successfully")

# Unpersist cache when done
df_dim_database.unpersist()

In [ ]:
df = spark.sql("SELECT * FROM SMD_LH.dim_database LIMIT 1000")
display(df)

In [ ]:
df = spark.sql("SELECT * FROM SMD_LH.tables LIMIT 1000")
display(df)